Extrator de Visuais — Power BI PDF

Recorta automaticamente gráficos e tabelas de painéis Power BI exportados em PDF.

**Fluxo:**
1. Converte cada página do PDF em imagem (alta resolução)
2. Detecta automaticamente os blocos visuais por bordas/contornos
3. Salva cada visual como imagem PNG separada

---

## 1. Instalação de Dependências

> **Poppler** também é necessário para o `pdf2image`:
> - **Windows** → Baixe em https://github.com/oschwartz10612/poppler-windows/releases e extraia em alguma pasta
> - **macOS** → `brew install poppler`
> - **Linux** → `sudo apt install poppler-utils`

In [ ]:
# Instala as bibliotecas Python necessárias
%pip install pdf2image opencv-python Pillow --quiet

## 2. Configurações

**Edite as variáveis abaixo antes de executar o restante do notebook.**

In [ ]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
ARQUIVO_PDF   = r'arquivo_pdf'          # ← seu arquivo PDF
PASTA_SAIDA   = r'pasta_saida'          # ← onde salvar os recortes
POPPLER_PATH  = r'pasta_bin' # ← pasta bin do Poppler (Windows)
                                                                        #   deixe None no macOS/Linux

# ── Resolução ─────────────────────────────────────────────────────────────────
DPI = 200   # 200 = boa qualidade | 300 = alta qualidade (mais lento)

# ── Filtros de detecção ───────────────────────────────────────────────────────
AREA_MINIMA    = 5000   # área mínima do visual (px²) — aumente para ignorar elementos pequenos
LARGURA_MINIMA = 80     # largura mínima (px)
ALTURA_MINIMA  = 60     # altura mínima (px)
PADDING        = 10     # margem extra ao redor de cada recorte (px)

# ── Debug ─────────────────────────────────────────────────────────────────────
SALVAR_DEBUG = True     # salva uma imagem mostrando os boxes detectados por página

print("Configurações carregadas.")

## 3. Importações

In [ ]:
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
from IPython.display import display, Image as IPImage
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso.")

## 4. Funções Principais

In [ ]:
def detectar_visuais(imagem_pil: Image.Image):
    """
    Detecta blocos visuais na imagem via contornos OpenCV.
    Retorna: (lista de boxes (x,y,w,h), imagem em formato OpenCV)
    """
    img_cv = cv2.cvtColor(np.array(imagem_pil), cv2.COLOR_RGB2BGR)
    gray   = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)

    # Threshold: fundo branco vira preto; bordas ficam evidentes
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)

    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in contornos:
        x, y, w, h = cv2.boundingRect(c)
        if w * h >= AREA_MINIMA and w >= LARGURA_MINIMA and h >= ALTURA_MINIMA:
            boxes.append((x, y, w, h))

    # Ordena: de cima para baixo, esquerda para direita
    boxes.sort(key=lambda b: (b[1], b[0]))
    return boxes, img_cv


def recortar_e_salvar(imagem_pil: Image.Image, boxes: list, pasta: Path, prefixo: str):
    """Recorta cada visual detectado e salva como PNG."""
    w_img, h_img = imagem_pil.size
    arquivos = []

    for i, (x, y, w, h) in enumerate(boxes, start=1):
        x1 = max(0, x - PADDING)
        y1 = max(0, y - PADDING)
        x2 = min(w_img, x + w + PADDING)
        y2 = min(h_img, y + h + PADDING)

        recorte = imagem_pil.crop((x1, y1, x2, y2))
        caminho = pasta / f"{prefixo}_visual_{i:02d}.png"
        recorte.save(str(caminho), "PNG")
        print(f" {caminho.name}  ({x2-x1} × {y2-y1} px)")
        arquivos.append(caminho)

    return arquivos


def salvar_debug(img_cv, boxes, pasta: Path, prefixo: str):
    """Salva imagem com os bounding boxes destacados em verde."""
    debug = img_cv.copy()
    h_img, w_img = debug.shape[:2]
    for x, y, w, h in boxes:
        x1, y1 = max(0, x - PADDING), max(0, y - PADDING)
        x2, y2 = min(w_img, x + w + PADDING), min(h_img, y + h + PADDING)
        cv2.rectangle(debug, (x1, y1), (x2, y2), (0, 180, 0), 2)
    caminho = pasta / f"{prefixo}_debug_boxes.png"
    cv2.imwrite(str(caminho), debug)
    print(f"Debug salvo: {caminho.name}")
    return caminho


print("Funções definidas.")

## 5. Processar PDF

Execute a célula abaixo para iniciar a extração.

In [ ]:
from pdf2image import convert_from_path

pdf   = Path(ARQUIVO_PDF)
saida = Path(PASTA_SAIDA)
saida.mkdir(parents=True, exist_ok=True)

# No macOS/Linux com Poppler no PATH, remova o argumento poppler_path
poppler_args = {"poppler_path": POPPLER_PATH} if POPPLER_PATH else {}

print(f"Convertendo '{pdf.name}' com {DPI} DPI...")
paginas = convert_from_path(str(pdf), dpi=DPI, **poppler_args)
print(f"   {len(paginas)} página(s) encontrada(s)\n")

todos_arquivos = []
debug_imagens  = []

for num_pag, pagina in enumerate(paginas, start=1):
    prefixo = f"pag{num_pag:02d}"
    print(f"── Página {num_pag} {'─'*40}")
    
    boxes, img_cv = detectar_visuais(pagina)
    print(f"   {len(boxes)} visual(is) detectado(s)")

    arquivos = recortar_e_salvar(pagina, boxes, saida, prefixo)
    todos_arquivos.extend(arquivos)

    if SALVAR_DEBUG:
        dbg = salvar_debug(img_cv, boxes, saida, prefixo)
        debug_imagens.append(dbg)

print(f"Concluído! {len(todos_arquivos)} visual(is) salvo(s) em '{saida}/'")

## 6. Visualizar Imagens de Debug

Mostra as imagens com os bounding boxes detectados para cada página.

In [ ]:
if SALVAR_DEBUG and debug_imagens:
    for i, caminho_debug in enumerate(debug_imagens, start=1):
        print(f"Página {i} — boxes detectados:")
        img_preview = Image.open(caminho_debug)
        # Reduz para exibição no notebook
        largura_max = 900
        if img_preview.width > largura_max:
            ratio = largura_max / img_preview.width
            novo_tamanho = (largura_max, int(img_preview.height * ratio))
            img_preview = img_preview.resize(novo_tamanho, Image.LANCZOS)
        display(img_preview)
else:
    print("Debug desativado ou nenhuma imagem gerada. Ative SALVAR_DEBUG = True na célula de configurações.")

## 7.  Visualizar Recortes Extraídos

Mostra todos os visuais extraídos diretamente no notebook.

In [ ]:
if todos_arquivos:
    print(f"Exibindo {len(todos_arquivos)} visual(is) extraído(s):\n")
    for caminho in todos_arquivos:
        print(f"{caminho.name}")
        img = Image.open(caminho)
        display(img)
        print()
else:
    print("Nenhum visual extraído. Verifique as configurações de detecção (AREA_MINIMA, etc.).")

## 8. Ajuste Fino dos Parâmetros

Se a detecção não estiver boa, ajuste os valores abaixo e reprocesse **uma única página** para testar rapidamente.

In [ ]:
# ── Parâmetros para ajuste fino ───────────────────────────────────────────────
PAGINA_TESTE   = 1      # número da página para testar (começa em 1)
AREA_MINIMA    = 5000   # aumente para filtrar elementos pequenos/ruído
LARGURA_MINIMA = 80     # largura mínima em pixels
ALTURA_MINIMA  = 60     # altura mínima em pixels
PADDING        = 10     # margem ao redor de cada recorte

# ── Teste em uma página ───────────────────────────────────────────────────────
pagina = paginas[PAGINA_TESTE - 1]
boxes, img_cv = detectar_visuais(pagina)

print(f"Página {PAGINA_TESTE}: {len(boxes)} visual(is) detectado(s)")
print("Bounding boxes (x, y, largura, altura):")
for i, b in enumerate(boxes, 1):
    print(f"  Visual {i:02d}: x={b[0]}, y={b[1]}, w={b[2]}, h={b[3]}, área={b[2]*b[3]}px²")

# Visualiza os boxes na página de teste
debug_test = img_cv.copy()
h_img, w_img = debug_test.shape[:2]
for x, y, w, h in boxes:
    x1, y1 = max(0, x - PADDING), max(0, y - PADDING)
    x2, y2 = min(w_img, x + w + PADDING), min(h_img, y + h + PADDING)
    cv2.rectangle(debug_test, (x1, y1), (x2, y2), (0, 180, 0), 2)

img_preview = Image.fromarray(cv2.cvtColor(debug_test, cv2.COLOR_BGR2RGB))
largura_max = 900
if img_preview.width > largura_max:
    ratio = largura_max / img_preview.width
    img_preview = img_preview.resize((largura_max, int(img_preview.height * ratio)), Image.LANCZOS)

display(img_preview)

## 9.  Processar Imagem Diretamente (sem PDF)

Se você já tiver um screenshot ou imagem PNG/JPG do painel, use esta célula.

In [ ]:
CAMINHO_IMAGEM = r'C:\Users\ramal\Documents\painel.png'  # ← caminho da imagem

img   = Image.open(CAMINHO_IMAGEM).convert("RGB")
saida = Path(PASTA_SAIDA)
saida.mkdir(parents=True, exist_ok=True)

print(f"🔄 Processando imagem '{Path(CAMINHO_IMAGEM).name}'...")
print(f"   Tamanho: {img.size[0]} × {img.size[1]} px\n")

boxes, img_cv = detectar_visuais(img)
print(f"   {len(boxes)} visual(is) detectado(s)\n")

arquivos = recortar_e_salvar(img, boxes, saida, "img")

if SALVAR_DEBUG:
    salvar_debug(img_cv, boxes, saida, "img")

print(f"{len(arquivos)} visual(is) salvo(s) em '{saida}/'")

# Exibe os recortes
for caminho in arquivos:
    print(f"{caminho.name}")
    display(Image.open(caminho))

---

##  Referência Rápida de Parâmetros

| Parâmetro | Descrição | Quando ajustar |
|---|---|---|
| `DPI` | Resolução da conversão PDF→imagem | Aumente para PDFs com texto pequeno |
| `AREA_MINIMA` | Área mínima de um visual (px²) | Aumente se aparecer lixo/ruído |
| `LARGURA_MINIMA` | Largura mínima (px) | Aumente para ignorar elementos estreitos |
| `ALTURA_MINIMA` | Altura mínima (px) | Aumente para ignorar linhas finas |
| `PADDING` | Margem extra ao redor do recorte (px) | Aumente se o visual estiver cortado |
| `SALVAR_DEBUG` | Salva imagem com boxes verdes | Útil para entender o que foi detectado |